<a href="https://colab.research.google.com/github/Satyalla42/semih-satya-wedding/blob/main/wedding_playlist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import requests
import base64
import json
import numpy as np
from itertools import permutations

# --- Step 1: Exchange auth code for access token ---
CLIENT_ID = '8fd99879d8724e97965efc1fe772d154'
CLIENT_SECRET = 'c7aed60e38ce488d9d5e098526d3312b'
REDIRECT_URI = 'https://open.spotify.com'
AUTH_CODE = 'AQA5hDLl3lDDf_OOyX5OkbSCDidxBQvIKQXou9UWpndLRZaorjTWEBkBejoC7296YxSW0wBw3IerxLNq5vtFMMviRkLukLDMKD2rvgYtW6b-G6Igj5koN4XvPudfFZPgquFcQzJTKZbxIzisEKGVBkaaLCW2MEzYfxgHmAqggUrzW4J-8B9v6hdHWUt-2qC4ogueesKGObxxJU4LuyXZYNxVBhBsEupM_GS87TNQe1ZSnNsun9deiG4wr-D1Oo3fXOVfFw'
PLAYLIST_ID = '79XTsRZqu4YZUi01Q1Gl5o'

credentials = base64.b64encode(f'{CLIENT_ID}:{CLIENT_SECRET}'.encode()).decode()
token_response = requests.post(
    'https://accounts.spotify.com/api/token',
    headers={'Authorization': f'Basic {credentials}', 'Content-Type': 'application/x-www-form-urlencoded'},
    data={'grant_type': 'authorization_code', 'code': AUTH_CODE, 'redirect_uri': REDIRECT_URI}
)
token_data = token_response.json()
print('Token response:', token_data.get('token_type'), '- expires in:', token_data.get('expires_in'))
ACCESS_TOKEN = token_data['access_token']
print('Got access token!')

Token response: Bearer - expires in: 3600
Got access token!


In [2]:
import urllib.parse

CLIENT_ID = '8fd99879d8724e97965efc1fe772d154'
REDIRECT_URI = 'https://open.spotify.com'
SCOPES = 'playlist-read-private playlist-modify-public playlist-modify-private'

auth_url = (
    'https://accounts.spotify.com/authorize?'
    + urllib.parse.urlencode({
        'client_id': CLIENT_ID,
        'response_type': 'code',
        'redirect_uri': REDIRECT_URI,
        'scope': SCOPES,
    })
)

print('Open this URL in your browser and authorize the app:')
print(auth_url)

Open this URL in your browser and authorize the app:
https://accounts.spotify.com/authorize?client_id=8fd99879d8724e97965efc1fe772d154&response_type=code&redirect_uri=https%3A%2F%2Fopen.spotify.com&scope=playlist-read-private+playlist-modify-public+playlist-modify-private


In [6]:
import requests
import numpy as np
from collections import defaultdict

HEADERS = {'Authorization': f'Bearer {ACCESS_TOKEN}'}
PLAYLIST_ID = '79XTsRZqu4YZUi01Q1Gl5o'

# --- Fetch all tracks ---
tracks = []
url = f'https://api.spotify.com/v1/playlists/{PLAYLIST_ID}/tracks?limit=100'
while url:
    r = requests.get(url, headers=HEADERS).json()
    for item in r['items']:
        t = item.get('track')
        if t and t.get('id'):
            tracks.append({
                'id': t['id'],
                'name': t['name'],
                'artist': t['artists'][0]['name'],
                'artist_id': t['artists'][0]['id'],
                'popularity': t.get('popularity', 0),
                'explicit': int(t.get('explicit', False)),
                'duration_ms': t.get('duration_ms', 0),
                'release_year': int(t.get('album', {}).get('release_date', '2000')[:4]),
            })
    url = r.get('next')

print(f'Fetched {len(tracks)} tracks')

# --- Fetch artist genres in batches of 50 ---
artist_ids = list(set(t['artist_id'] for t in tracks))
artist_genres = {}
for i in range(0, len(artist_ids), 50):
    batch = ','.join(artist_ids[i:i+50])
    r = requests.get(f'https://api.spotify.com/v1/artists?ids={batch}', headers=HEADERS).json()
    for artist in r.get('artists', []):
        if artist:
            artist_genres[artist['id']] = artist.get('genres', [])

# Collect all unique genres
all_genres = sorted(set(g for genres in artist_genres.values() for g in genres))
print(f'Found {len(all_genres)} unique genres across artists')

# --- Build feature vectors ---
# Genre one-hot + normalized popularity, explicit, duration, release_year
genre_index = {g: i for i, g in enumerate(all_genres)}
n_genres = len(all_genres)

def build_vector(track):
    genre_vec = np.zeros(n_genres)
    for g in artist_genres.get(track['artist_id'], []):
        if g in genre_index:
            genre_vec[genre_index[g]] = 1.0

    meta = np.array([
        track['popularity'] / 100.0,
        track['explicit'],
        track['duration_ms'] / 600000.0,  # normalize to ~10 min max
        (track['release_year'] - 1950) / 80.0,  # normalize year
    ])
    # Weight: genres matter most, then metadata
    return np.concatenate([genre_vec * 3.0, meta])

vectors = {t['id']: build_vector(t) for t in tracks}
print('Built feature vectors')

# --- Nearest-neighbor greedy sort ---
remaining = [t['id'] for t in tracks]
sorted_ids = [remaining.pop(0)]

while remaining:
    current = sorted_ids[-1]
    cv = vectors[current]
    best_id = min(remaining, key=lambda tid: np.linalg.norm(cv - vectors[tid]))
    sorted_ids.append(best_id)
    remaining.remove(best_id)

id_to_track = {t['id']: t for t in tracks}
print("\nSorted playlist order:")
for i, tid in enumerate(sorted_ids):
    t = id_to_track[tid]
    genres = artist_genres.get(t['artist_id'], [])[:2]
    print(f"{i+1:3}. {t['artist']} - {t['name']} [{', '.join(genres)}]")

Fetched 148 tracks
Found 66 unique genres across artists
Built feature vectors

Sorted playlist order:
  1. The Goo Goo Dolls - Iris []
  2. Natasha Bedingfield - Unwritten []
  3. B.o.B - Nothin' on You (feat. Bruno Mars) []
  4. Coldplay - Paradise []
  5. Black Eyed Peas - I Gotta Feeling []
  6. Macklemore - Can't Hold Us (feat. Ray Dalton) []
  7. Nico & Vinz - Am I Wrong []
  8. John Newman - Love Me Again []
  9. Bruno Mars - Marry You []
 10. Kings of Leon - Use Somebody []
 11. Gym Class Heroes - Stereo Hearts (feat. Adam Levine) []
 12. The Script - Hall of Fame (feat. will.i.am) []
 13. Capital Cities - Safe and Sound []
 14. Twenty One Pilots - Stressed Out []
 15. Twenty One Pilots - Ride []
 16. Bastille - Pompeii []
 17. Elle King - Ex's & Oh's []
 18. B.o.B - So Good []
 19. Family of the Year - Hero []
 20. James Morrison - Broken Strings []
 21. Coldplay - Viva La Vida []
 22. Alex Gaudino - Destination Calabria - Radio Edit []
 23. Toploader - Dancing in the Moonligh

In [7]:
# --- Apply the sorted order to the Spotify playlist ---
# The Spotify API can only replace up to 100 tracks at a time with PUT,
# so we first replace the first 100, then add the rest.
# For playlists > 100 tracks we use PUT (first 100) + POST (remaining)

HEADERS = {'Authorization': f'Bearer {ACCESS_TOKEN}', 'Content-Type': 'application/json'}
PLAYLIST_ID = '79XTsRZqu4YZUi01Q1Gl5o'

track_uris = [f'spotify:track:{tid}' for tid in sorted_ids]

# Step 1: Replace entire playlist with first 100 tracks
first_batch = track_uris[:100]
r = requests.put(
    f'https://api.spotify.com/v1/playlists/{PLAYLIST_ID}/tracks',
    headers=HEADERS,
    json={'uris': first_batch}
)
print(f'PUT first 100 tracks: {r.status_code}')

# Step 2: Append remaining tracks in batches of 100
for i in range(100, len(track_uris), 100):
    batch = track_uris[i:i+100]
    r = requests.post(
        f'https://api.spotify.com/v1/playlists/{PLAYLIST_ID}/tracks',
        headers=HEADERS,
        json={'uris': batch}
    )
    print(f'POST tracks {i+1}-{min(i+100, len(track_uris))}: {r.status_code}')

print(f'\nDone! Playlist reordered with {len(sorted_ids)} tracks.')
print('The songs are now sorted so similar-sounding tracks are next to each other!')

PUT first 100 tracks: 200
POST tracks 101-148: 201

Done! Playlist reordered with 148 tracks.
The songs are now sorted so similar-sounding tracks are next to each other!


In [10]:
# Manual genre overrides for all artists missing genre data
# Keyed by artist name for easy lookup

manual_genres = {
    # Pop / Indie Pop / Soft Pop
    'Bruno Mars':           ['pop', 'r&b'],
    'Justin Bieber':        ['pop', 'teen pop'],
    'Shawn Mendes':         ['pop', 'soft pop'],
    'Natasha Bedingfield':  ['pop', 'soft pop'],
    'Madonna':              ['pop', 'dance pop'],
    'Lana Del Rey':         ['dream pop', 'indie pop'],
    'Myles Smith':          ['soft pop', 'indie pop'],
    'Alex Warren':          ['pop', 'indie pop'],
    'CYRIL':                ['pop', 'indie pop'],
    'Family of the Year':   ['indie pop', 'folk pop'],
    'Eagle-Eye Cherry':     ['soft pop', 'alternative rock'],
    'Scott Thomas':         ['pop', 'indie pop'],

    # Alternative Rock / Indie Rock
    'Coldplay':             ['alternative rock', 'post-britpop'],
    'Kings of Leon':        ['alternative rock', 'indie rock'],
    'The Cranberries':      ['alternative rock', 'irish rock'],
    'Bastille':             ['indie pop', 'alternative rock'],
    'Twenty One Pilots':    ['alternative rock', 'indie pop'],
    'Wheatus':              ['pop punk', 'alternative rock'],
    'Reamonn':              ['alternative rock', 'soft rock'],
    'Hurts':                ['synth-pop', 'alternative rock'],
    'Capital Cities':       ['indie pop', 'alternative rock'],
    'Lighthouse Family':    ['soft rock', 'soul'],

    # New Wave / Synth Pop / 80s
    'The Police':           ['new wave', 'rock'],
    'La Roux':              ['synth-pop', 'new wave'],
    'Kim Wilde':            ['synth-pop', 'new wave'],
    'Laura Branigan':       ['synth-pop', 'new wave'],

    # Pop Rock / Soft Rock / Classic Pop
    'The Goo Goo Dolls':    ['pop rock', 'alternative rock'],
    'James Morrison':       ['soft rock', 'pop rock'],
    'Nico & Vinz':          ['pop', 'afropop'],
    'John Newman':          ['pop', 'soul'],
    'The Script':           ['pop rock', 'soft rock'],
    'Elle King':            ['alternative rock', 'blues rock'],
    'Cher':                 ['pop', 'dance pop'],
    'Connor Price':         ['pop', 'hip hop'],
    'Steven Rodriguez':     ['pop', 'r&b'],

    # Soul / R&B / Funk
    'Michael Jackson':      ['pop', 'r&b', 'soul'],
    'Whitney Houston':      ['r&b', 'soul', 'pop'],
    'Cass Elliot':          ['folk pop', 'soft rock'],

    # Dance / Electronic / House
    'Alex Gaudino':         ['eurodance', 'italo dance', 'dance pop'],
    'Eric Prydz':           ['progressive house', 'electro house'],
    'Rui Da Silva':         ['progressive house', 'trance'],
    'Niconé':               ['melodic techno', 'house'],
    'Klangkarussell':       ['tropical house', 'deep house'],
    'Fritz Kalkbrenner':    ['minimal techno', 'techno'],
    'Wamdue Project':       ['house', 'deep house'],

    # Rock / Classic Rock
    'Survivor':             ['classic rock', 'hard rock'],
    'The Outfield':         ['classic rock', 'power pop'],
    'Men At Work':          ['new wave', 'classic rock'],
    'Alannah Myles':        ['blues rock', 'classic rock'],
    'Marc Cohn':            ['soft rock', 'soul'],
    'Mungo Jerry':          ['classic rock', 'folk rock'],
    'Cutting Crew':         ['soft rock', 'new wave'],
    'Black':                ['soft rock', 'pop'],

    # Pop / Hip Hop / Urban
    'B.o.B':                ['hip hop', 'pop rap'],
    'Macklemore':           ['hip hop', 'pop rap'],
    'Black Eyed Peas':      ['hip hop', 'dance pop'],
    'Gym Class Heroes':     ['hip hop', 'pop punk'],

    # Folk / Indie
    'Toploader':            ['britpop', 'indie rock'],
    'Fools Garden':         ['britpop', 'indie pop'],
    'Beagle Music':         ['indie pop', 'folk pop'],

    # Other
    'Madonna':              ['pop', 'dance pop'],
}

# Apply overrides: update artist_genres for artists with no genres
overrides_applied = 0
for t in tracks:
    artist_name = t['artist']
    if not artist_genres.get(t['artist_id'], []) and artist_name in manual_genres:
        artist_genres[t['artist_id']] = manual_genres[artist_name]
        overrides_applied += 1

# Recheck
still_missing = [(t['artist'], t['name']) for t in tracks if not artist_genres.get(t['artist_id'], [])]
print(f"Applied overrides, {overrides_applied} artist IDs updated")
print(f"Still missing genres: {len(still_missing)}")
for a, n in still_missing:
    print(f"  {a} - {n}")

Applied overrides, 60 artist IDs updated
Still missing genres: 0


In [11]:
# Rebuild genre index and vectors with full genre data, then re-sort and apply

all_genres = sorted(set(g for genres in artist_genres.values() for g in genres))
genre_index = {g: i for i, g in enumerate(all_genres)}
n_genres = len(all_genres)
print(f"Total unique genres now: {n_genres}")

def build_vector(track):
    genre_vec = np.zeros(n_genres)
    for g in artist_genres.get(track['artist_id'], []):
        if g in genre_index:
            genre_vec[genre_index[g]] = 1.0
    meta = np.array([
        track['popularity'] / 100.0,
        track['explicit'],
        track['duration_ms'] / 600000.0,
        (track['release_year'] - 1950) / 80.0,
    ])
    return np.concatenate([genre_vec * 3.0, meta])

vectors = {t['id']: build_vector(t) for t in tracks}

# Nearest-neighbor greedy sort
remaining = [t['id'] for t in tracks]
sorted_ids = [remaining.pop(0)]
while remaining:
    current = sorted_ids[-1]
    cv = vectors[current]
    best_id = min(remaining, key=lambda tid: np.linalg.norm(cv - vectors[tid]))
    sorted_ids.append(best_id)
    remaining.remove(best_id)

id_to_track = {t['id']: t for t in tracks}
print("\nNew sorted playlist order:")
for i, tid in enumerate(sorted_ids):
    t = id_to_track[tid]
    genres = artist_genres.get(t['artist_id'], [])[:2]
    print(f"{i+1:3}. {t['artist']} - {t['name']} [{', '.join(genres)}]")

Total unique genres now: 84

New sorted playlist order:
  1. The Goo Goo Dolls - Iris [pop rock, alternative rock]
  2. The Goo Goo Dolls - Here Is Gone [pop rock, alternative rock]
  3. The Killers - Mr. Brightside [alternative rock]
  4. Kings of Leon - Sex on Fire [alternative rock, indie rock]
  5. Kings of Leon - Use Somebody [alternative rock, indie rock]
  6. Bastille - Pompeii [indie pop, alternative rock]
  7. Capital Cities - Safe and Sound [indie pop, alternative rock]
  8. Twenty One Pilots - Stressed Out [alternative rock, indie pop]
  9. Twenty One Pilots - Ride [alternative rock, indie pop]
 10. Elle King - Ex's & Oh's [alternative rock, blues rock]
 11. Coldplay - Viva La Vida [alternative rock, post-britpop]
 12. Coldplay - Paradise [alternative rock, post-britpop]
 13. Wheatus - Teenage Dirtbag [pop punk, alternative rock]
 14. Eagle-Eye Cherry - Save Tonight [soft pop, alternative rock]
 15. OneRepublic - Stop And Stare [soft pop]
 16. OneRepublic - All The Right Mov

In [12]:
# Apply new sorted order to Spotify
HEADERS = {'Authorization': f'Bearer {ACCESS_TOKEN}', 'Content-Type': 'application/json'}
PLAYLIST_ID = '79XTsRZqu4YZUi01Q1Gl5o'
track_uris = [f'spotify:track:{tid}' for tid in sorted_ids]

r = requests.put(
    f'https://api.spotify.com/v1/playlists/{PLAYLIST_ID}/tracks',
    headers=HEADERS,
    json={'uris': track_uris[:100]}
)
print(f'PUT first 100 tracks: {r.status_code}')

for i in range(100, len(track_uris), 100):
    batch = track_uris[i:i+100]
    r = requests.post(
        f'https://api.spotify.com/v1/playlists/{PLAYLIST_ID}/tracks',
        headers=HEADERS,
        json={'uris': batch}
    )
    print(f'POST tracks {i+1}-{min(i+100, len(track_uris))}: {r.status_code}')

print(f'\nDone! Playlist reordered with full genre data ({len(sorted_ids)} tracks).')

PUT first 100 tracks: 200
POST tracks 101-148: 201

Done! Playlist reordered with full genre data (148 tracks).


In [13]:
import numpy as np

# ── Step 1: Assign an energy score (0.0–1.0) per genre ──────────────────────
genre_energy = {
    # Very low – romantic / ambient / slow ballads
    'soul': 0.15, 'doo-wop': 0.15, 'soft rock': 0.20, 'folk rock': 0.20,
    'folk pop': 0.20, 'dream pop': 0.20, 'bedroom pop': 0.20,
    'baroque pop': 0.22, 'soft pop': 0.25, 'christmas': 0.25,
    'indie pop': 0.30, 'jangle pop': 0.30, 'dansk pop': 0.30,

    # Low-medium – chill pop / classic rock mellow side
    'pop': 0.35, 'r&b': 0.35, 'yacht rock': 0.35, 'aor': 0.35,
    'teen pop': 0.35, 'dance pop': 0.40, 'afropop': 0.38,
    'britpop': 0.40, 'madchester': 0.42, 'post-britpop': 0.40,
    'latin pop': 0.38, 'latin': 0.38, 'latin alternative': 0.38,
    'german pop': 0.40,

    # Medium – mid-energy rock / pop rock / indie
    'pop rock': 0.45, 'classic rock': 0.50, 'blues rock': 0.50,
    'southern rock': 0.50, 'surf rock': 0.52, 'jangle pop': 0.45,
    'psychedelic rock': 0.48, 'irish rock': 0.50, 'finnish rock': 0.50,
    'alternative rock': 0.55, 'indie rock': 0.55, 'post-grunge': 0.55,
    'rock': 0.55, 'new wave': 0.55, 'synth-pop': 0.55, 'synthpop': 0.55,

    # Medium-high – danceable / energetic
    'pop punk': 0.65, 'punk': 0.65, 'hardcore punk': 0.68,
    'glam rock': 0.60, 'glam metal': 0.65, 'hard rock': 0.65,
    'progressive rock': 0.58, 'funk rock': 0.65,
    'hip hop': 0.62, 'pop rap': 0.62, 'german hip hop': 0.62,
    'french pop': 0.50, 'neue deutsche welle': 0.50,
    'schlager': 0.55, 'schlagerparty': 0.60,
    'reggae': 0.58, 'ragga': 0.60, 'dancehall': 0.62,

    # High – dance floor / electronic
    'eurodance': 0.80, 'europop': 0.75, 'italo dance': 0.80,
    'dance pop': 0.72, 'edm': 0.82, 'tropical house': 0.70,
    'deep house': 0.72, 'house': 0.78, 'french house': 0.78,
    'disco house': 0.78, 'progressive house': 0.80,
    'electro house': 0.82, 'trance': 0.80,
    'eurodance': 0.82, 'minimal techno': 0.78, 'techno': 0.80,
    'melodic techno': 0.78,
}

def track_energy(track):
    genres = artist_genres.get(track['artist_id'], [])
    if not genres:
        return 0.45
    scores = [genre_energy.get(g, 0.45) for g in genres]
    return np.mean(scores)

energy_map = {t['id']: track_energy(t) for t in tracks}

# ── Step 2: Define target wedding energy arc ────────────────────────────────
# Shape: gentle start → slow build → peak → soft landing
# Segments (as fraction of playlist length):
#   0–15%:  0.20→0.35  (dinner / arrival, soft)
#   15–35%: 0.35→0.55  (warming up)
#   35–65%: 0.55→0.85  (peak party)
#   65–85%: 0.85→0.65  (sustained high energy)
#   85–100%:0.65→0.30  (wind down)
N = len(tracks)
def target_energy(pos):
    t = pos / (N - 1)
    if t < 0.15:
        return 0.20 + (0.35 - 0.20) * (t / 0.15)
    elif t < 0.35:
        return 0.35 + (0.55 - 0.35) * ((t - 0.15) / 0.20)
    elif t < 0.65:
        return 0.55 + (0.85 - 0.55) * ((t - 0.35) / 0.30)
    elif t < 0.85:
        return 0.85 - (0.85 - 0.65) * ((t - 0.65) / 0.20)
    else:
        return 0.65 - (0.65 - 0.30) * ((t - 0.85) / 0.15)

target_curve = np.array([target_energy(i) for i in range(N)])
print("Energy arc defined. Target shape:")
for i in [0, 22, 52, 97, 125, 147]:
    print(f"  position {i:3d}: target energy = {target_curve[i]:.2f}")

Energy arc defined. Target shape:
  position   0: target energy = 0.20
  position  22: target energy = 0.35
  position  52: target energy = 0.55
  position  97: target energy = 0.84
  position 125: target energy = 0.65
  position 147: target energy = 0.30


In [14]:
# ── Step 3 & 4: Try all starting points, score against energy arc, pick best ─

# Combined feature vector: genre similarity + energy-arc pull
# We blend the genre/meta distance with how well the energy at each step
# matches the target curve. Alpha controls how strongly the arc is enforced.
ALPHA = 2.0   # weight of energy-arc penalty vs genre distance

all_track_ids = [t['id'] for t in tracks]

def nn_sort_with_energy(start_id):
    remaining = [tid for tid in all_track_ids if tid != start_id]
    seq = [start_id]
    while remaining:
        pos = len(seq)
        current = seq[-1]
        cv = vectors[current]
        t_e = target_curve[pos]   # target energy at this position
        # Cost = genre distance + alpha * energy mismatch
        best_id = min(
            remaining,
            key=lambda tid: (
                np.linalg.norm(cv - vectors[tid])
                + ALPHA * abs(energy_map[tid] - t_e)
            )
        )
        seq.append(best_id)
        remaining.remove(best_id)
    return seq

def score_sequence(seq):
    # Mean squared error between actual energy curve and target
    actual = np.array([energy_map[tid] for tid in seq])
    return np.mean((actual - target_curve) ** 2)

print(f"Testing {len(all_track_ids)} starting points...")
best_score = float('inf')
best_seq = None
best_start = None

for i, start_id in enumerate(all_track_ids):
    seq = nn_sort_with_energy(start_id)
    score = score_sequence(seq)
    if score < best_score:
        best_score = score
        best_seq = seq
        best_start = start_id
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(all_track_ids)} done, best score so far: {best_score:.4f}")

sorted_ids = best_seq
id_to_track = {t['id']: t for t in tracks}
best_track = id_to_track[best_start]
print(f"\nBest starting track: {best_track['artist']} - {best_track['name']}")
print(f"Energy arc score (MSE): {best_score:.4f}  (lower = better fit)")
print("\nFinal sorted playlist:")
for i, tid in enumerate(sorted_ids):
    t = id_to_track[tid]
    e = energy_map[tid]
    genres = artist_genres.get(t['artist_id'], [])[:2]
    print(f"{i+1:3}. [E:{e:.2f}] {t['artist']} - {t['name']} [{', '.join(genres)}]")

Testing 148 starting points...
  25/148 done, best score so far: 0.0356
  50/148 done, best score so far: 0.0333
  75/148 done, best score so far: 0.0333
  100/148 done, best score so far: 0.0333
  125/148 done, best score so far: 0.0333

Best starting track: Phil Collins - Another Day in Paradise - 2016 Remaster
Energy arc score (MSE): 0.0333  (lower = better fit)

Final sorted playlist:
  1. [E:0.20] Phil Collins - Another Day in Paradise - 2016 Remaster [soft rock]
  2. [E:0.20] Sting - Englishman In New York [soft rock]
  3. [E:0.20] Sting - Fields Of Gold [soft rock]
  4. [E:0.20] Rod Stewart - Sailing [soft rock]
  5. [E:0.20] Cass Elliot - Make Your Own Kind Of Music [folk pop, soft rock]
  6. [E:0.20] Vance Joy - Riptide [folk pop]
  7. [E:0.20] Of Monsters and Men - Little Talks [folk pop]
  8. [E:0.25] Family of the Year - Hero [indie pop, folk pop]
  9. [E:0.25] Beagle Music - Like Ice in the Sunshine [indie pop, folk pop]
 10. [E:0.25] Lana Del Rey - Love [dream pop, indie 

In [16]:
# Check current playlist track count on Spotify
import requests
HEADERS = {'Authorization': f'Bearer {ACCESS_TOKEN}'}
PLAYLIST_ID = '79XTsRZqu4YZUi01Q1Gl5o'
r = requests.get(f'https://api.spotify.com/v1/playlists/{PLAYLIST_ID}', headers=HEADERS).json()
total = r['tracks']['total']
print(f"Tracks currently in playlist: {total}")
print(f"Tracks in sorted_ids: {len(sorted_ids)}")
print(f"Tracks in tracks list: {len(tracks)}")

Tracks currently in playlist: 148
Tracks in sorted_ids: 148
Tracks in tracks list: 148
